In [2]:
import os
import gc
import pandas as pd

def merge_main_with_bureau(
    main_path='/Users/nguyenminhtri/FinalYearPro/data/processed/df_main_clean_fe.parquet',
    bureau_path='/Users/nguyenminhtri/FinalYearPro/data/processed/df_bureau_clean_fe.parquet',
    output_path='/Users/nguyenminhtri/FinalYearPro/data/processed/df_main_x_bureau.parquet'
):
    """
    Hàm gộp df_main_fe và df_bureau_clean_fe theo SK_ID_CURR và lưu thành df_main_x_bureau.parquet
    """
    print("🚀 BẮT ĐẦU TIẾN HÀNH GỘP BẢNG MAIN x BUREAU...")
    print("-" * 60)

    # 1. Kiểm tra sự tồn tại của file
    if not os.path.exists(main_path):
        raise FileNotFoundError(f"❌ Không tìm thấy file Main tại: {main_path}")
    if not os.path.exists(bureau_path):
        raise FileNotFoundError(f"❌ Không tìm thấy file Bureau tại: {bureau_path}")

    # 2. Nạp dữ liệu
    print("📥 Đang nạp dữ liệu từ thư mục processed...")
    df_main = pd.read_parquet(main_path)
    df_bureau = pd.read_parquet(bureau_path)

    print(f"   • Main Dataset   : {df_main.shape[0]:,} dòng | {df_main.shape[1]} cột")
    print(f"   • Bureau Dataset : {df_bureau.shape[0]:,} dòng | {df_bureau.shape[1]} cột")

    # 3. Thực hiện Left Join theo SK_ID_CURR
    print("\n🔗 Đang thực hiện LEFT JOIN theo 'SK_ID_CURR'...")
    df_merged = df_main.merge(df_bureau, on='SK_ID_CURR', how='left')

    # 4. Lưu file Parquet kết quả
    os.makedirs(os.path.dirname(output_path), exist_ok=True)
    df_merged.to_parquet(output_path, index=False)

    print("=" * 60)
    print(f"📊 Final Dataset Shape: {df_merged.shape[0]:,} rows | {df_merged.shape[1]} columns")
    print(f"✨ Added Features    : +{df_merged.shape[1] - df_main.shape[1]} features from Bureau")
    print(f"💾 Saved successfully to: {output_path}")
    print("=" * 60)

    # 5. Dọn dẹp RAM
    del df_main, df_bureau, df_merged
    gc.collect()
    print("🧹 Memory cleared successfully!")

# --- THỰC THI HÀM GỘP ---
if __name__ == '__main__':
    merge_main_with_bureau()

🚀 BẮT ĐẦU TIẾN HÀNH GỘP BẢNG MAIN x BUREAU...
------------------------------------------------------------
📥 Đang nạp dữ liệu từ thư mục processed...
   • Main Dataset   : 307,511 dòng | 47 cột
   • Bureau Dataset : 305,811 dòng | 373 cột

🔗 Đang thực hiện LEFT JOIN theo 'SK_ID_CURR'...
📊 Final Dataset Shape: 307,511 rows | 419 columns
✨ Added Features    : +372 features from Bureau
💾 Saved successfully to: /Users/nguyenminhtri/FinalYearPro/data/processed/df_main_x_bureau.parquet
🧹 Memory cleared successfully!


In [3]:
import os
import gc
import pandas as pd

# ==============================================================================
# CELL MERGE TỔNG HỢP: BUREAU BALANCE (SK_ID_BUREAU) -> BUREAU CLEAN -> MAIN FE
# ==============================================================================

# 1. Đường dẫn các file đầu vào
main_path = '/data/processed/table/df_main_clean_fe.parquet'
bureau_path = '/data/processed/table/df_bureau_clean_fe.parquet'
bureau_bal_path = '/data/processed/table/df_bureau_balance_clean_fe.parquet'
raw_bureau_map_path = '/Users/nguyenminhtri/FinalYearPro/data/raw/bureau.csv'
output_path = '/Users/nguyenminhtri/FinalYearPro/data/processed/df_main_x_bureau_all.parquet'

print("=" * 70)
print("🚀 BẮT ĐẦU QUY TRÌNH MERGE TỪ DƯỚI LÊN (CASCADING MERGE)")
print("=" * 70)

# ------------------------------------------------------------------------------
# BƯỚC 1: MAP BUREAU BALANCE VỀ SK_ID_CURR & NÉN LÊN CẤP KHÁCH HÀNG
# ------------------------------------------------------------------------------
print("🔹 [1/3] Đang nạp Bureau Balance & mapping lên SK_ID_CURR...")

df_bb = pd.read_parquet(bureau_bal_path)
df_map = pd.read_csv(raw_bureau_map_path, usecols=['SK_ID_CURR', 'SK_ID_BUREAU'])

print(f"   • Bureau Balance gốc : {df_bb.shape[0]:,} khoản vay | {df_bb.shape[1]} cột")

# Map SK_ID_CURR và nén về 1 Khách hàng = 1 Dòng
df_bb_mapped = df_bb.merge(df_map, on='SK_ID_BUREAU', how='inner').drop(columns=['SK_ID_BUREAU'])

feature_cols = [c for c in df_bb_mapped.columns if c != 'SK_ID_CURR']
bb_aggs = {}
for col in feature_cols:
    if any(k in col for k in ['SUM', 'COUNT', 'RECENT']):
        bb_aggs[col] = ['sum', 'max', 'mean']
    else:
        bb_aggs[col] = ['mean', 'max']

df_bb_curr = df_bb_mapped.groupby('SK_ID_CURR').agg(bb_aggs)
df_bb_curr.columns = [f"{col}_{stat.upper()}" for col, stat in df_bb_curr.columns]
df_bb_curr.reset_index(inplace=True)

print(f"   ✅ Sau khi nén lên SK_ID_CURR: {df_bb_curr.shape[0]:,} khách hàng | {df_bb_curr.shape[1]} thuộc tính")

# Dọn dẹp RAM tạm
del df_bb, df_map, df_bb_mapped
gc.collect()

# ------------------------------------------------------------------------------
# BƯỚC 2 & 3: NẠP MAIN, BUREAU CLEAN & CASCADING LEFT JOIN
# ------------------------------------------------------------------------------
print("\n🔹 [2/3] Đang nạp df_main_fe và df_bureau_clean_fe...")
df_main = pd.read_parquet(main_path)
df_bureau = pd.read_parquet(bureau_path)

print(f"   • Main Dataset   : {df_main.shape[0]:,} dòng | {df_main.shape[1]} cột")
print(f"   • Bureau Dataset : {df_bureau.shape[0]:,} dòng | {df_bureau.shape[1]} cột")

print("\n🔹 [3/3] Đang tiến hành LEFT JOIN liên hoàn vào bảng Main...")
# Main x Bureau
df_final = df_main.merge(df_bureau, on='SK_ID_CURR', how='left')

# Main/Bureau x Bureau Balance
df_final = df_final.merge(df_bb_curr, on='SK_ID_CURR', how='left')

# ------------------------------------------------------------------------------
# LƯU FILE KẾT QUẢ CUỐI CÙNG
# ------------------------------------------------------------------------------
os.makedirs(os.path.dirname(output_path), exist_ok=True)
df_final.to_parquet(output_path, index=False)

print("\n" + "=" * 70)
print("🎉 TẠO BẢNG TRAIN TỔNG HỢP THÀNH CÔNG!")
print("=" * 70)
print(f"📊 Kích thước bảng train mới : {df_final.shape[0]:,} dòng | {df_final.shape[1]} cột")
print(f"✨ Số thuộc tính đã tăng     : +{df_final.shape[1] - df_main.shape[1]} features")
print(f"💾 Saved to                  : {output_path}")
print("=" * 70)

# Dọn dẹp RAM
del df_main, df_bureau, df_bb_curr, df_final
gc.collect()
print("🧹 Memory cleared successfully!")

🚀 BẮT ĐẦU QUY TRÌNH MERGE TỪ DƯỚI LÊN (CASCADING MERGE)
🔹 [1/3] Đang nạp Bureau Balance & mapping lên SK_ID_CURR...
   • Bureau Balance gốc : 817,395 khoản vay | 33 cột
   ✅ Sau khi nén lên SK_ID_CURR: 134,542 khách hàng | 80 thuộc tính

🔹 [2/3] Đang nạp df_main_fe và df_bureau_clean_fe...
   • Main Dataset   : 307,511 dòng | 47 cột
   • Bureau Dataset : 305,811 dòng | 373 cột

🔹 [3/3] Đang tiến hành LEFT JOIN liên hoàn vào bảng Main...

🎉 TẠO BẢNG TRAIN TỔNG HỢP THÀNH CÔNG!
📊 Kích thước bảng train mới : 307,511 dòng | 498 cột
✨ Số thuộc tính đã tăng     : +451 features
💾 Saved to                  : /Users/nguyenminhtri/FinalYearPro/data/processed/df_main_x_bureau_all.parquet
🧹 Memory cleared successfully!


In [6]:
# ==============================================================================
# CELL MERGE INTEGRATION: MAIN x BUREAU ALL + PREVIOUS APPLICATION CLEAN FE
# ==============================================================================

import os
import gc
import pandas as pd


def merge_main_bureau_with_prev_app(
        main_bureau_path='/Users/nguyenminhtri/FinalYearPro/data/processed/df_main_x_bureau_all.parquet',
        prev_app_path='/Users/nguyenminhtri/FinalYearPro/data/processed/prev_app_clean_FE.parquet',
        output_path='/Users/nguyenminhtri/FinalYearPro/data/processed/df_main_x_bureau_x_prev.parquet'
):
    """
    Hàm gộp bảng df_main_x_bureau_all với bảng prev_app_clean_FE đã ép phẳng theo khóa SK_ID_CURR.
    """
    print("=" * 70)
    print("🚀 BẮT ĐẦU TIẾN HÀNH MERGE PREVIOUS APPLICATION VÀO BẢNG TỔNG HỢP")
    print("=" * 70)

    # 1. Kiểm tra sự tồn tại của file
    if not os.path.exists(main_bureau_path):
        raise FileNotFoundError(f"❌ Không tìm thấy file Main x Bureau tại: {main_bureau_path}")
    if not os.path.exists(prev_app_path):
        raise FileNotFoundError(f"❌ Không tìm thấy file Previous Application tại: {prev_app_path}")

    # 2. Nạp dữ liệu
    print("📥 Đang nạp dữ liệu từ thư mục processed...")
    df_main_bureau = pd.read_parquet(main_bureau_path)
    df_prev_app = pd.read_parquet(prev_app_path)

    print(
        f"   • Current Main Dataset (Main x Bureau All) : {df_main_bureau.shape[0]:,} dòng | {df_main_bureau.shape[1]} cột")
    print(f"   • Previous Application Clean FE Dataset    : {df_prev_app.shape[0]:,} dòng | {df_prev_app.shape[1]} cột")

    # 3. Thực hiện Left Join theo khóa SK_ID_CURR
    print("\n🔗 Đang thực hiện LEFT JOIN theo 'SK_ID_CURR'...")
    df_merged = df_main_bureau.merge(df_prev_app, on='SK_ID_CURR', how='left')

    # 4. Lưu file Parquet kết quả chuẩn hóa
    os.makedirs(os.path.dirname(output_path), exist_ok=True)
    df_merged.to_parquet(output_path, index=False)

    print("\n" + "=" * 70)
    print("🎉 TẠO BẢNG TỔNG HỢP MAIN x BUREAU x PREVIOUS APPLICATION THÀNH CÔNG!")
    print("=" * 70)
    print(f"📊 Final Dataset Shape : {df_merged.shape[0]:,} rows | {df_merged.shape[1]} columns")
    print(f"✨ Added Features      : +{df_merged.shape[1] - df_main_bureau.shape[1]} features from Previous Application")
    print(f"💾 Saved successfully  : {output_path}")
    print("=" * 70)

    # 5. Dọn dẹp RAM
    del df_main_bureau, df_prev_app, df_merged
    gc.collect()
    print("🧹 Memory cleared successfully!")


# --- THỰC THI HÀM GỘP ---
if __name__ == '__main__':
    merge_main_bureau_with_prev_app()

🚀 BẮT ĐẦU TIẾN HÀNH MERGE PREVIOUS APPLICATION VÀO BẢNG TỔNG HỢP
📥 Đang nạp dữ liệu từ thư mục processed...
   • Current Main Dataset (Main x Bureau All) : 307,511 dòng | 498 cột
   • Previous Application Clean FE Dataset    : 338,857 dòng | 593 cột

🔗 Đang thực hiện LEFT JOIN theo 'SK_ID_CURR'...

🎉 TẠO BẢNG TỔNG HỢP MAIN x BUREAU x PREVIOUS APPLICATION THÀNH CÔNG!
📊 Final Dataset Shape : 307,511 rows | 1091 columns
✨ Added Features      : +593 features from Previous Application
💾 Saved successfully  : /Users/nguyenminhtri/FinalYearPro/data/processed/df_main_x_bureau_x_prev.parquet
🧹 Memory cleared successfully!


In [14]:
import os
import gc
import pandas as pd

# 1. Khai báo chính xác đường dẫn tuyệt đối theo đúng cây thư mục trong ảnh
pos_file_path = '/Users/nguyenminhtri/FinalYearPro/data/processed/table/pos_cash_clean_FE.parquet'
prev_merged_path = '/Users/nguyenminhtri/FinalYearPro/data/processed/df_main_x_bureau_x_prev.parquet'
output_merged_path = '/Users/nguyenminhtri/FinalYearPro/data/processed/df_main_x_bureau_x_prev_x_pos.parquet'

print("🚀 Đang đọc dữ liệu theo đúng cấu trúc thư mục...")
df_merged = pd.read_parquet(prev_merged_path)
df_pos = pd.read_parquet(pos_file_path)

# Đảm bảo SK_ID_CURR nằm ở dạng Cột (reset index nếu kẹt ở index)
if 'SK_ID_CURR' not in df_merged.columns and df_merged.index.name == 'SK_ID_CURR':
    df_merged = df_merged.reset_index()

if 'SK_ID_CURR' not in df_pos.columns and df_pos.index.name == 'SK_ID_CURR':
    df_pos = df_pos.reset_index()

print(f"📊 Bảng tổng cũ (df_main_x_bureau_x_prev) : {df_merged.shape[0]:,} dòng | {df_merged.shape[1]} cột")
print(f"📊 Bảng POS mới (pos_cash_clean_FE)      : {df_pos.shape[0]:,} dòng | {df_pos.shape[1]} cột")

# Ghép thủ công bằng LEFT JOIN qua SK_ID_CURR
print("🔗 Đang thực hiện ghép trực tiếp...")
df_merged = df_merged.merge(df_pos, on='SK_ID_CURR', how='left')

# Lưu file kết quả
df_merged.to_parquet(output_merged_path, index=False)

print("=" * 75)
print(f"🎉 THÀNH CÔNG! Đã lưu tệp tổng mới tại:\n👉 {output_merged_path}")
print(f"📊 Dimensions tệp tổng mới : {df_merged.shape[0]:,} dòng | {df_merged.shape[1]} cột")
print("=" * 75)

# Giải phóng bộ nhớ RAM
del df_pos
gc.collect()

🚀 Đang đọc dữ liệu theo đúng cấu trúc thư mục...
📊 Bảng tổng cũ (df_main_x_bureau_x_prev) : 307,511 dòng | 1091 cột
📊 Bảng POS mới (pos_cash_clean_FE)      : 337,252 dòng | 95 cột
🔗 Đang thực hiện ghép trực tiếp...
🎉 THÀNH CÔNG! Đã lưu tệp tổng mới tại:
👉 /Users/nguyenminhtri/FinalYearPro/data/processed/df_main_x_bureau_x_prev_x_pos.parquet
📊 Dimensions tệp tổng mới : 307,511 dòng | 1185 cột


0

In [15]:
# --- CELL 9: MANUAL LEFT-JOIN INSTALLMENTS_PAYMENTS INTO MASTER DATASET ---

import os
import gc
import pandas as pd

# 1. Khai báo đường dẫn tuyệt đối theo đúng cây thư mục workspace
ins_file_path = '/Users/nguyenminhtri/FinalYearPro/data/processed/table/installments_payments_clean_FE.parquet'
prev_merged_path = '/Users/nguyenminhtri/FinalYearPro/data/processed/df_main_x_bureau_x_prev_x_pos.parquet'
output_merged_path = '/Users/nguyenminhtri/FinalYearPro/data/processed/df_main_x_bureau_x_prev_x_pos_x_ins.parquet'

print("🚀 Đang tiến hành ghép file installments_payments vào tệp tổng hợp...")

# 2. Đọc 2 file parquet
df_master = pd.read_parquet(prev_merged_path)
df_ins_agg = pd.read_parquet(ins_file_path)

# Đảm bảo cột khóa SK_ID_CURR nằm ở dạng Cột chuẩn (reset index nếu bị kẹt)
if 'SK_ID_CURR' not in df_master.columns and df_master.index.name == 'SK_ID_CURR':
    df_master = df_master.reset_index()

if 'SK_ID_CURR' not in df_ins_agg.columns and df_ins_agg.index.name == 'SK_ID_CURR':
    df_ins_agg = df_ins_agg.reset_index()

print(f"📊 Kích thước Master Dataset cũ : {df_master.shape[0]:,} dòng | {df_master.shape[1]} cột")
print(f"📊 Kích thước Bảng INS mới      : {df_ins_agg.shape[0]:,} dòng | {df_ins_agg.shape[1]} cột")

# 3. Ghép trực tiếp bằng LEFT JOIN qua SK_ID_CURR
print("🔗 Đang thực hiện ghép trực tiếp theo SK_ID_CURR...")
df_master = df_master.merge(df_ins_agg, on='SK_ID_CURR', how='left')

# 4. Xuất file Parquet master mới nhất
df_master.to_parquet(output_merged_path, index=False)

print("=" * 75)
print(f"🎉 THÀNH CÔNG! Master dataset mới đã được lưu tại:\n👉 {output_merged_path}")
print(f"📊 Dimensions Master Dataset mới : {df_master.shape[0]:,} dòng | {df_master.shape[1]} cột")
print("=" * 75)

# Giải phóng bộ nhớ RAM
del df_ins_agg
gc.collect()

🚀 Đang tiến hành ghép file installments_payments vào tệp tổng hợp...
📊 Kích thước Master Dataset cũ : 307,511 dòng | 1185 cột
📊 Kích thước Bảng INS mới      : 339,587 dòng | 53 cột
🔗 Đang thực hiện ghép trực tiếp theo SK_ID_CURR...
🎉 THÀNH CÔNG! Master dataset mới đã được lưu tại:
👉 /Users/nguyenminhtri/FinalYearPro/data/processed/df_main_x_bureau_x_prev_x_pos_x_ins.parquet
📊 Dimensions Master Dataset mới : 307,511 dòng | 1237 cột


0

In [19]:
# --- CELL: MERGE FEATURE-ENGINEERED SUB-TABLES WITH APPLICATION_TRAIN ---

import os
import time
import pandas as pd

start_time = time.time()
print("🚀 Bắt đầu quá trình Merge dữ liệu tổng...")

# 1. Đường dẫn tuyệt đối đến bảng gốc (File Parquet)
app_train_path = '/Users/nguyenminhtri/FinalYearPro/data/processed/df_main_x_bureau_x_prev_x_pos_x_ins.parquet'

# 2. Đọc bảng chính bằng pd.read_parquet (THAY CHO pd.read_csv)
print(f"⏳ Đang đọc bảng gốc (Parquet)...")
df_master = pd.read_parquet(app_train_path)
initial_rows = df_master.shape[0]
print(f"   • Bảng gốc: {initial_rows:,} dòng | {df_master.shape[1]} cột")

# 3. Danh sách ĐƯỜNG DẪN TUYỆT ĐỐI của các bảng phụ đã ép phẳng
sub_table_paths = [
    '/Users/nguyenminhtri/FinalYearPro/data/processed/table/credit_card_balance_aggregated.parquet',
]

# 4. Vòng lặp Left Join từng bảng phụ vào bảng chính
for table_path in sub_table_paths:
    table_name = os.path.basename(table_path)

    if os.path.exists(table_path):
        print(f"⏳ Đang Left Join với bảng phụ: {table_name}...")
        df_sub = pd.read_parquet(table_path)

        # Kiểm tra sự tồn tại của khóa SK_ID_CURR
        if 'SK_ID_CURR' in df_sub.columns:
            cols_to_use = df_sub.columns.difference(df_master.columns).tolist() + ['SK_ID_CURR']
            df_master = df_master.merge(df_sub[cols_to_use], on='SK_ID_CURR', how='left')
            print(f"   ✓ Đã gộp thành công! Số cột hiện tại: {df_master.shape[1]}")
        else:
            print(f"   ⚠️ CẢNH BÁO: Bảng {table_name} không có cột 'SK_ID_CURR'. Bỏ qua!")
    else:
        print(f"   ⚠️ CẢNH BÁO: Không tìm thấy file tại {table_path}. Bỏ qua!")

# 5. Kiểm tra tính toàn vẹn dữ liệu sau khi merge
assert df_master.shape[
           0] == initial_rows, f"❌ LỖI: Số dòng bị thay đổi từ {initial_rows:,} thành {df_master.shape[0]:,}!"
print("✅ Kiểm tra toàn vẹn thành công: Số dòng không bị nhân bản/thất thoát!")

# 6. Đường dẫn tuyệt đối lưu file Master
output_master_path = '/Users/nguyenminhtri/FinalYearPro/data/processed/master_train_dataset.parquet'
os.makedirs(os.path.dirname(output_master_path), exist_ok=True)
df_master.to_parquet(output_master_path, compression='snappy', index=False)

exec_time = time.time() - start_time

print("=" * 75)
print(f"🎉 HOÀN TẤT MERGE DỮ LIỆU TỔNG TRONG: {exec_time:.2f} giây")
print(f"💾 File Master đã lưu tại : {output_master_path}")
print(f"📊 Kích thước bảng Master : {df_master.shape[0]:,} khách hàng | {df_master.shape[1]} cột")
print(f"💾 Dung lượng RAM sử dụng : {df_master.memory_usage().sum() / 1024 ** 2:.2f} MB")
print("=" * 75)

🚀 Bắt đầu quá trình Merge dữ liệu tổng...
⏳ Đang đọc bảng gốc (Parquet)...
   • Bảng gốc: 307,511 dòng | 1237 cột
⏳ Đang Left Join với bảng phụ: credit_card_balance_aggregated.parquet...
   ✓ Đã gộp thành công! Số cột hiện tại: 1380
✅ Kiểm tra toàn vẹn thành công: Số dòng không bị nhân bản/thất thoát!
🎉 HOÀN TẤT MERGE DỮ LIỆU TỔNG TRONG: 10.52 giây
💾 File Master đã lưu tại : /Users/nguyenminhtri/FinalYearPro/data/processed/master_train_dataset.parquet
📊 Kích thước bảng Master : 307,511 khách hàng | 1380 cột
💾 Dung lượng RAM sử dụng : 3257.94 MB


In [21]:
# --- CELL: AUTOMATED MASTER DATASET MERGING PIPELINE (AUTO-FIX INDEX) ---

import os
import time
import pandas as pd

start_time = time.time()
print("=" * 80)
print("🚀 BẮT ĐẦU CHUỖI TIẾN TRÌNH GỘP DỮ LIỆU TỔNG (MASTER MERGE PIPELINE)")
print("=" * 80)

# 1. Khai báo đường dẫn tuyệt đối
MAIN_PATH = '/Users/nguyenminhtri/FinalYearPro/data/processed/df_main_x_bureau_all.parquet'

SUB_TABLES = [
    '/Users/nguyenminhtri/FinalYearPro/data/processed/table/prev_app_clean_fe_v2.parquet',
    '/Users/nguyenminhtri/FinalYearPro/data/processed/table/pos_cash_clean_FE.parquet',
    '/Users/nguyenminhtri/FinalYearPro/data/processed/table/installments_payments_clean_FE.parquet',
    '/Users/nguyenminhtri/FinalYearPro/data/processed/table/credit_card_balance_aggregated.parquet'
]

OUTPUT_MASTER = '/Users/nguyenminhtri/FinalYearPro/data/processed/master_train_dataset_v2.parquet'

# 2. Đọc bảng chính
print(f"⏳ Đang nạp bảng chính từ: {MAIN_PATH}")
df_master = pd.read_parquet(MAIN_PATH) if MAIN_PATH.endswith('.parquet') else pd.read_csv(MAIN_PATH)

if 'SK_ID_CURR' not in df_master.columns and df_master.index.name == 'SK_ID_CURR':
    df_master = df_master.reset_index()

initial_rows = df_master.shape[0]
print(f"   • Bảng gốc ban đầu: {initial_rows:,} khách hàng | {df_master.shape[1]} cột")
print("-" * 80)

# 3. Vòng lặp gộp từng bảng phụ với Auto-Fix Index
for i, table_path in enumerate(SUB_TABLES, 1):
    table_name = os.path.basename(table_path)

    if not os.path.exists(table_path):
        print(f"⚠️ [Bảng {i}/{len(SUB_TABLES)}] Không tìm thấy file tại '{table_path}'. BỎ QUA!")
        continue

    print(f"⏳ [Bảng {i}/{len(SUB_TABLES)}] Đang nạp và Left Join với: {table_name}...")
    df_sub = pd.read_parquet(table_path) if table_path.endswith('.parquet') else pd.read_csv(table_path)

    # KỂM TRA & TỰ ĐỘNG KÉO SK_ID_CURR TỪ INDEX NẾU BỊ KẸT
    if 'SK_ID_CURR' not in df_sub.columns:
        if df_sub.index.name == 'SK_ID_CURR' or 'SK_ID_CURR' in str(df_sub.index.names):
            df_sub = df_sub.reset_index()
            print(f"   💡 Đã tự động khôi phục cột 'SK_ID_CURR' từ Index cho {table_name}!")
        else:
            print(f"   ⚠️ CẢNH BÁO: Bảng {table_name} hoàn toàn không có 'SK_ID_CURR'. BỎ QUA!")
            continue

    # Lấy danh sách cột mới (tránh trùng cột đã có trong df_master)
    cols_to_use = df_sub.columns.difference(df_master.columns).tolist() + ['SK_ID_CURR']

    # Left Join
    df_master = df_master.merge(df_sub[cols_to_use], on='SK_ID_CURR', how='left')
    print(f"   ✓ Gộp thành công! Tổng số cột hiện tại: {df_master.shape[1]}")

print("-" * 80)

# 4. Kiểm tra toàn vẹn & Lưu file
assert df_master.shape[0] == initial_rows, f"❌ LỖI: Số dòng bị thay đổi!"
print("✅ KIỂM TRA TOÀN VẸN THÀNH CÔNG: Số lượng khách hàng giữ nguyên 100%!")

os.makedirs(os.path.dirname(OUTPUT_MASTER), exist_ok=True)
df_master.to_parquet(OUTPUT_MASTER, compression='snappy', index=False)

exec_time = time.time() - start_time
print("=" * 80)
print(f"🎉 HOÀN TẤT MERGE TẤT CẢ CÁC BẢNG TRONG: {exec_time:.2f} GIÂY")
print(f"📊 KÍCH THƯỚC BẢNG MASTER : {df_master.shape[0]:,} khách hàng | {df_master.shape[1]} thuộc tính")
print(f"💾 DUNG LƯỢNG RAM SỬ DỤNG  : {df_master.memory_usage().sum() / 1024 ** 2:.2f} MB")
print("=" * 80)

🚀 BẮT ĐẦU CHUỖI TIẾN TRÌNH GỘP DỮ LIỆU TỔNG (MASTER MERGE PIPELINE)
⏳ Đang nạp bảng chính từ: /Users/nguyenminhtri/FinalYearPro/data/processed/df_main_x_bureau_all.parquet
   • Bảng gốc ban đầu: 307,511 khách hàng | 498 cột
--------------------------------------------------------------------------------
⏳ [Bảng 1/4] Đang nạp và Left Join với: prev_app_clean_fe_v2.parquet...
   💡 Đã tự động khôi phục cột 'SK_ID_CURR' từ Index cho prev_app_clean_fe_v2.parquet!
   ✓ Gộp thành công! Tổng số cột hiện tại: 743
⏳ [Bảng 2/4] Đang nạp và Left Join với: pos_cash_clean_FE.parquet...
   💡 Đã tự động khôi phục cột 'SK_ID_CURR' từ Index cho pos_cash_clean_FE.parquet!
   ✓ Gộp thành công! Tổng số cột hiện tại: 837
⏳ [Bảng 3/4] Đang nạp và Left Join với: installments_payments_clean_FE.parquet...
   ✓ Gộp thành công! Tổng số cột hiện tại: 889
⏳ [Bảng 4/4] Đang nạp và Left Join với: credit_card_balance_aggregated.parquet...
   ✓ Gộp thành công! Tổng số cột hiện tại: 1032
--------------------------------